In [3]:
!pip install -q kagglehub torch torchvision ultralytics

In [1]:
import os
from pathlib import Path

# ── Project root = directory containing this notebook ──
PROJECT_DIR = Path(os.getcwd())
print(f"Project directory: {PROJECT_DIR}")

# ── Create all needed directories ──
DIRS = [
    "data/raw",
    "data/raw/24_chromosomes_object/JEPG",
    "data/raw/24_chromosomes_object/annotations",
    "data/train/images", "data/train/labels",
    "data/val/images",   "data/val/labels",
    "data/test/images",   "data/test/labels",
    "runs",
]

for d in DIRS:
    p = PROJECT_DIR / d
    p.mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {p}")

print("\nAll directories created.")

Project directory: /home/krstff/unica_project
  ✓ /home/krstff/unica_project/data/raw
  ✓ /home/krstff/unica_project/data/raw/24_chromosomes_object/JEPG
  ✓ /home/krstff/unica_project/data/raw/24_chromosomes_object/annotations
  ✓ /home/krstff/unica_project/data/train/images
  ✓ /home/krstff/unica_project/data/train/labels
  ✓ /home/krstff/unica_project/data/val/images
  ✓ /home/krstff/unica_project/data/val/labels
  ✓ /home/krstff/unica_project/data/test/images
  ✓ /home/krstff/unica_project/data/test/labels
  ✓ /home/krstff/unica_project/runs

All directories created.


In [2]:
import os
from pathlib import Path

# ── Kaggle authentication ──
# Option A: Set environment variables (recommended for Colab / cloud)
#   export KAGGLE_USERNAME='your_username'
#   export KAGGLE_KEY='your_key'
# Option B: Place kaggle.json in ~/.kaggle/ or set KAGGLE_CONFIG_DIR

if not os.environ.get("KAGGLE_USERNAME") and not os.environ.get("KAGGLE_KEY"):
    # Try to find kaggle.json in the project directory as a fallback
    kaggle_json = Path("kaggle.json")
    if kaggle_json.exists():
        config_dir = Path.home() / ".kaggle"
        config_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(kaggle_json, config_dir / "kaggle.json")
        os.chmod(config_dir / "kaggle.json", 0o600)
        print("Copied kaggle.json to ~/.kaggle/")
    else:
        print("⚠️  No Kaggle credentials found.")
        print("   Set KAGGLE_USERNAME + KAGGLE_KEY env vars, or")
        print("   place kaggle.json in the project directory.")
else:
    print("Kaggle credentials found via environment variables.")

Copied kaggle.json to ~/.kaggle/


In [3]:
import kagglehub
from pathlib import Path
import shutil

PROJECT_DIR = Path(os.getcwd())

# Download the dataset
print("Downloading chromosome dataset from Kaggle (~990 MB)...")
download_path = kagglehub.dataset_download(
    "aliabedimadiseh/chromosome-image-dataset-karyotype"
)
print(f"Downloaded to: {download_path}")

# Copy data into our project structure
RAW_DIR = PROJECT_DIR / "data/raw/24_chromosomes_object"
source_data = Path(download_path) / "Data/24_chromosomes_object"

if not source_data.exists():
    # Kagglehub may have nested differently; try alternate path
    source_data = Path(download_path) / "24_chromosomes_object"

print(f"Source data dir: {source_data}")
print(f"Destination dir: {RAW_DIR}")

# Copy images
jpeg_src = source_data / "JEPG"
jpeg_dst = RAW_DIR / "JEPG"
if jpeg_src.exists():
    shutil.copytree(jpeg_src, jpeg_dst, dirs_exist_ok=True)
    print(f"Copied images: {len(list(jpeg_dst.glob('*.jpg')))} files")
else:
    print("⚠️  JEPG directory not found in dataset!")

# Copy annotations
ann_src = source_data / "annotations"
ann_dst = RAW_DIR / "annotations"
if ann_src.exists():
    shutil.copytree(ann_src, ann_dst, dirs_exist_ok=True)
    print(f"Copied annotations: {len(list(ann_dst.glob('*.xml')))} files")

# Copy split files (train.txt, test.txt, diff_image.txt)
for txt_file in ["train.txt", "test.txt", "diff_image.txt"]:
    src = Path(download_path) / "Data" / txt_file
    dst = PROJECT_DIR / "data/raw" / txt_file
    if src.exists():
        shutil.copy(src, dst)
        print(f"Copied {txt_file}")

print("\n✅ Dataset ready in data/raw/")

Source data dir: /home/krstff/.cache/kagglehub/datasets/aliabedimadiseh/chromosome-image-dataset-karyotype/versions/4/Data/24_chromosomes_object
Destination dir: /home/krstff/unica_project/data/raw/24_chromosomes_object
Copied images: 5000 files
Copied annotations: 5000 files
Copied train.txt
Copied test.txt
Copied diff_image.txt

✅ Dataset ready in data/raw/


In [4]:
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
RAW_DIR      = PROJECT_DIR / "data/raw"

# Paths inside raw data
images_dir  = RAW_DIR / "24_chromosomes_object/JEPG"
annotations_dir = RAW_DIR / "24_chromosomes_object/annotations"

# Destination directories
SPLITS = {
    "train": PROJECT_DIR / "data/train",
    "val":   PROJECT_DIR / "data/val",
    "test":  PROJECT_DIR / "data/test",
}

# Ensure all split dirs exist
for split_dir in SPLITS.values():
    (split_dir / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "labels").mkdir(parents=True, exist_ok=True)

# Read split lists
def read_split(filepath):
    if filepath.exists():
        return {line.strip() for line in filepath.read_text().splitlines() if line.strip()}
    return set()

train_files = read_split(RAW_DIR / "train.txt")
test_files  = read_split(RAW_DIR / "test.txt")
val_files   = read_split(RAW_DIR / "diff_image.txt")

print(f"Split sizes: train={len(train_files)}, test={len(test_files)}, val={len(val_files)}")

# Discover all class names from annotations & build mapping
all_class_names = set()
for xml_file in annotations_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    for obj in tree.getroot().findall("object"):
        all_class_names.add(obj.find("name").text)

# alphabetical -> deterministic IDs
CLASS_NAMES = sorted(all_class_names)
CLASS_MAP   = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print(f"\nFound {len(CLASS_NAMES)} classes:")
for name, idx in CLASS_MAP.items():
    print(f"  {idx}: {name}")

# XML -> YOLO converter
def xml_to_yolo(xml_path, img_width, img_height):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall("object"):
        cls_name = obj.find("name").text
        cls_id   = CLASS_MAP.get(cls_name, 0)

        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        # Skip degenerate boxes
        if xmax <= xmin or ymax <= ymin:
            continue

        x_center = (xmin + xmax) / 2.0 / img_width
        y_center = (ymin + ymax) / 2.0 / img_height
        w = (xmax - xmin) / img_width
        h = (ymax - ymin) / img_height

        lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
    return "\n".join(lines)

# Distribute files into splits
counts = {s: 0 for s in SPLITS}
missing_images = 0
missing_annots = 0
conversions = 0

all_filenames = train_files | test_files | val_files
print(f"\nTotal unique files to process: {len(all_filenames)}")

for filename in all_filenames:
    if filename in val_files:
        split = "val"
    elif filename in test_files:
        split = "test"
    else:
        split = "train"

    img_src = images_dir / f"{filename}.jpg"
    ann_src = annotations_dir / f"{filename}.xml"
    split_dir = SPLITS[split]

    # Copy image
    if img_src.exists():
        shutil.copy2(img_src, split_dir / "images" / f"{filename}.jpg")
    else:
        missing_images += 1
        continue

    # Convert & write YOLO label
    if ann_src.exists():
        from PIL import Image
        try:
            with Image.open(img_src) as img:
                w, h = img.size
        except Exception:
            tree = ET.parse(ann_src)
            root = tree.getroot()
            size = root.find("size")
            w = float(size.find("width").text) if size is not None else 640
            h = float(size.find("height").text) if size is not None else 640

        yolo_text = xml_to_yolo(ann_src, w, h)
        label_dst = split_dir / "labels" / f"{filename}.txt"
        label_dst.write_text(yolo_text + "\n")
        conversions += 1
    else:
        missing_annots += 1
        (split_dir / "labels" / f"{filename}.txt").write_text("")

    counts[split] += 1

print(f"\nDone!")
print(f"   Converted {conversions} XML -> YOLO labels")
for split, count in counts.items():
    print(f"   {split}: {count} images")
if missing_images:
    print(f"   WARNING: {missing_images} images missing (skipped)")
if missing_annots:
    print(f"   WARNING: {missing_annots} annotations missing (empty labels created)")


Split sizes: train=3750, test=1250, val=122

Found 24 classes:
  0: A1
  1: A2
  2: A3
  3: B4
  4: B5
  5: C10
  6: C11
  7: C12
  8: C6
  9: C7
  10: C8
  11: C9
  12: D13
  13: D14
  14: D15
  15: E16
  16: E17
  17: E18
  18: F19
  19: F20
  20: G21
  21: G22
  22: X
  23: Y

Total unique files to process: 5000

Done!
   Converted 4994 XML -> YOLO labels
   train: 3744 images
   val: 122 images
   test: 1128 images


In [5]:
from pathlib import Path
import xml.etree.ElementTree as ET

PROJECT_DIR = Path(os.getcwd()).resolve()

# Build class names list (same order as Cell 4)
all_class_names = set()
ann_dir = PROJECT_DIR / "data/raw/24_chromosomes_object/annotations"
for xml_file in ann_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    for obj in tree.getroot().findall("object"):
        all_class_names.add(obj.find("name").text)

CLASS_NAMES = sorted(all_class_names)
nc = len(CLASS_NAMES)

# Build YAML content
lines = []
lines.append(f"# YOLOv8 dataset config - chromosome detection ({nc} classes)")
lines.append(f"path: {PROJECT_DIR / 'data'}")
lines.append("train: train/images")
lines.append("val: val/images")
lines.append("test: test/images")
lines.append("")
lines.append(f"nc: {nc}")
names_str = ", ".join([f"'{n}'" for n in CLASS_NAMES])
lines.append(f"names: [{names_str}]")

yaml_content = "\n".join(lines) + "\n"

yaml_path = PROJECT_DIR / "mydataset.yaml"
yaml_path.write_text(yaml_content)

print(f"Written: {yaml_path}")
print("\nContents:")
print(yaml_content)


Written: /home/krstff/unica_project/mydataset.yaml

Contents:
# YOLOv8 dataset config - chromosome detection (24 classes)
path: /home/krstff/unica_project/data
train: train/images
val: val/images
test: test/images

nc: 24
names: ['A1', 'A2', 'A3', 'B4', 'B5', 'C10', 'C11', 'C12', 'C6', 'C7', 'C8', 'C9', 'D13', 'D14', 'D15', 'E16', 'E17', 'E18', 'F19', 'F20', 'G21', 'G22', 'X', 'Y']



In [6]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(os.getcwd())

# Load pretrained YOLOv8 nano DETECTION model (not segmentation - no mask data in XML)
model = YOLO("yolov8n.pt")

# Train - adjust batch_size if you get OOM errors
results = model.train(
    data=str(PROJECT_DIR / "mydataset.yaml"),
    epochs=100,
    batch=8,
    imgsz=640,
    verbose=True,
    plots=True,
    workers=4,
    project=str(PROJECT_DIR / "runs"),  # keep runs inside project
    name="train",
    patience=20,      # early stopping if val loss doesn't improve
)

# Print where the best model was saved
best_wts = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nBest model saved to: {best_wts}")


Ultralytics 8.4.82 🚀 Python-3.14.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060, 11912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/krstff/unica_project/mydataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto,

In [9]:
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
best_model_path = PROJECT_DIR / "runs/train-3/weights/best.pt"

# Reload best model for validation
model = YOLO(str(best_model_path))

metrics = model.val(
    data=str(PROJECT_DIR / "mydataset.yaml"),
    split="val",
)

print(f"mAP50:  {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")


Ultralytics 8.4.82 🚀 Python-3.14.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3060, 11912MiB)
Model summary (fused): 73 layers, 3,010,328 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 104.8±23.6 MB/s, size: 112.4 KB)
val: Scanning /home/krstff/unica_project/data/val/labels.cache... 122 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 122/122 227.6Kit/s 0.0s
val: /home/krstff/unica_project/data/val/images/1051851.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.3s/it 10.6s0.3s
                   all        122       5615      0.946      0.921      0.972      0.754
                    A1        122        244      0.974      0.984      0.994      0.842
                    A2        122        244      0.956      0.984      0.993      0.846
                    A3        122        244      0.975      0.926      0.988      0.822
               

In [8]:
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR = Path(os.getcwd())
best_model_path = PROJECT_DIR / "runs/train-3/weights/best.pt"

# Load best model
model = YOLO(str(best_model_path))

# Pick a test image for inference
test_images_dir = PROJECT_DIR / "data/test/images"
test_files = list(test_images_dir.glob("*.jpg"))

if not test_files:
    print("No test images found. Using a train image instead.")
    test_files = list((PROJECT_DIR / "data/train/images").glob("*.jpg"))

sample_image = test_files[0]
print(f"Running inference on: {sample_image.name}")

# Predict
results = model.predict(str(sample_image), conf=0.3, save=True)

# Display the result
saved_img = Path(results[0].save_dir) / f"{sample_image.name}"

from PIL import Image
img = Image.open(saved_img)
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis("off")
plt.title(f"Prediction on {sample_image.name}")
plt.tight_layout()
plt.show()


Running inference on: 1050224.jpg

image 1/1 /home/krstff/unica_project/data/test/images/1050224.jpg: 640x608 2 A1s, 2 A2s, 2 A3s, 2 B4s, 2 B5s, 2 C10s, 2 C11s, 2 C12s, 2 C6s, 2 C7s, 2 C8s, 2 C9s, 2 D13s, 2 D14s, 2 D15s, 2 E16s, 2 E17s, 2 E18s, 2 F19s, 2 F20s, 2 G21s, 2 G22s, 2 Xs, 121.7ms
Speed: 24.2ms preprocess, 121.7ms inference, 26.9ms postprocess per image at shape (1, 3, 640, 608)
Results saved to /home/krstff/unica_project/runs/detect/predict


<Figure size 1200x800 with 1 Axes>